In [15]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import StepLR
from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Union
import networkx as nx
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from sklearn.cluster import KMeans

In [16]:
df = pd.read_csv('C:\\Users\\ekru\\Documents\\Cursor\\ganv2\\Project\\Dataset Creation\\Final_Graph.csv')
print(df.head())

                          Head        Relation           Tail
0       Fluocinolone acetonide  INTERACTS_WITH       Abetimus
1                   Tazarotene  INTERACTS_WITH     Oxybutynin
2                    Guanoclor  INTERACTS_WITH  Nortriptyline
3  Dexamethasone isonicotinate  INTERACTS_WITH    Oxaliplatin
4                 Sparfloxacin  INTERACTS_WITH   Iron sucrose


In [17]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [18]:
print("Train:")
print(train_df)
print("Test:")
print(test_df)
print("Validation:")
print(valid_df)

Train:
                                                   Head         Relation  \
194639                                      Venlafaxine   INTERACTS_WITH   
338072                               Dexmethylphenidate       IN_PRODUCT   
413921                                    Nortriptyline   INTERACTS_WITH   
276174                                      Prenylamine   INTERACTS_WITH   
439150                                   Isosulfan blue   INTERACTS_WITH   
...                                                 ...              ...   
259178                                          Demadex  MANUFACTURED_BY   
365838                                         Restoril        HAS_PRICE   
131932  Discount Drug Mart nausea control Cherry Flavor  MANUFACTURED_BY   
146867                                          AVE9633   INTERACTS_WITH   
121958                                         Doxapram   INTERACTS_WITH   

                                     Tail  
194639                  Magnesium su

In [19]:
#Preprocessing the data

# Strip whitespace from column names and values
for df in [train_df, test_df, valid_df]:
    df.columns = df.columns.str.strip()
    df['Head'] = df['Head'].str.strip()
    df['Tail'] = df['Tail'].str.strip()
    df['Relation'] = df['Relation'].str.strip()

# Map entities and relations to indices
entities = pd.concat([train_df['Head'], train_df['Tail'],
                      test_df['Head'], test_df['Tail'],
                      valid_df['Head'], valid_df['Tail']]).unique()
relations = pd.concat([train_df['Relation'],
                       test_df['Relation'],
                       valid_df['Relation']]).unique()

entity_to_idx = {entity: idx for idx, entity in enumerate(entities)}
relation_to_idx = {relation: idx for idx, relation in enumerate(relations)}

# Function to convert a dataframe to a torch_geometric Data object
def create_graph_data(df):
    edge_index = torch.tensor([
        [entity_to_idx[head] for head in df['Head']],
        [entity_to_idx[tail] for tail in df['Tail']]
    ], dtype=torch.long)
    
    edge_attr = torch.tensor([relation_to_idx[rel] for rel in df['Relation']], dtype=torch.long)
    
    data = Data(edge_index=edge_index, edge_attr=edge_attr)
    data.num_nodes = len(entity_to_idx)
    return data

train_data = create_graph_data(train_df)
valid_data = create_graph_data(valid_df)
test_data  = create_graph_data(test_df)


In [20]:
#Model Creation
class AttRGCN(torch.nn.Module):
    def __init__(self, num_entities, num_relations, hidden_dim, num_layers=2, dropout=0.3, num_bases=30, num_heads=4):
        super(AttRGCN, self).__init__()

        self.entity_embedding = torch.nn.Embedding(num_entities, hidden_dim)
        torch.nn.init.xavier_uniform_(self.entity_embedding.weight)  # Better initialization
        
        self.rgcn_layers = torch.nn.ModuleList()
        for _ in range(num_layers):
            conv = RGCNConv(hidden_dim, hidden_dim, num_relations, num_bases=num_bases)
            self.rgcn_layers.append(conv)
        
        self.norm = torch.nn.LayerNorm(hidden_dim)
        self.dropout = torch.nn.Dropout(dropout)
        
        self.attention = torch.nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, kdim=hidden_dim, vdim=hidden_dim)
        self.attention_proj = torch.nn.Linear(hidden_dim, hidden_dim)
        
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim*2),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim*2, num_relations)
        )

    def forward(self, data):
        device = data.edge_index.device
        x = self.entity_embedding(torch.arange(data.num_nodes, device=device))
        
        for conv in self.rgcn_layers:
            x_new = conv(x, data.edge_index, data.edge_attr)
            x_new = self.norm(x + x_new)  # Skip connection
            x_new = F.gelu(x_new)
            x = self.dropout(x_new)
        
        head_emb = x[data.edge_index[0]]
        tail_emb = x[data.edge_index[1]]
        
        query = self.attention_proj(head_emb).unsqueeze(0)
        key = self.attention_proj(tail_emb).unsqueeze(0)
        value = (head_emb + tail_emb).unsqueeze(0)
        
        attn_output, _ = self.attention(query, key, value)
        edge_features = attn_output.squeeze(0) + head_emb  # Residual connection
        
        out = self.fc(edge_features)
        return F.log_softmax(out, dim=1)

In [21]:
#Training Configuration
hidden_dim = 256
num_layers = 3
dropout = 0.3
learning_rate = 0.0005
epochs = 50
patience = 3
weight_decay = 1e-4 

num_entities = len(entity_to_idx)
num_relations = len(relation_to_idx)

In [22]:
model = AttRGCN(num_entities, num_relations, hidden_dim, num_layers=num_layers,
                          dropout=dropout, num_bases=30, num_heads=4)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=patience)

In [23]:
train_losses = []
train_accuracies = []

best_val_loss = float('inf')
patience_counter = 0

In [10]:
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(train_data)
    loss = criterion(output, train_data.edge_attr)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
    optimizer.step()
    
    train_predictions = torch.argmax(output, dim=1)
    correct_predictions = (train_predictions.cpu().numpy() == train_data.edge_attr.cpu().numpy()).sum()
    train_accuracy = correct_predictions / len(train_data.edge_attr)
    train_losses.append(loss.item())
    train_accuracies.append(train_accuracy)
    
    # Evaluate on validation data
    model.eval()
    with torch.no_grad():
        val_output = model(valid_data)
        val_loss = criterion(val_output, valid_data.edge_attr)
    
    # Update scheduler based on validation loss
    scheduler.step(val_loss)
    print(f"Epoch {epoch+1:02d}: Train Loss = {loss.item():.4f}, Val Loss = {val_loss.item():.4f}")
    
    # Early stopping check
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        patience_counter = 0
        best_model_state = model.state_dict()
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print("Early stopping triggered.")
            break

# Load the best model obtained
model.load_state_dict(best_model_state)

Epoch 01: Train Loss = 1.8875, Val Loss = 1.2127
Epoch 02: Train Loss = 1.2643, Val Loss = 0.8142
Epoch 03: Train Loss = 0.8500, Val Loss = 0.6860
Epoch 04: Train Loss = 0.6718, Val Loss = 0.6796
Epoch 05: Train Loss = 0.6286, Val Loss = 0.6340
Epoch 06: Train Loss = 0.5682, Val Loss = 0.5490
Epoch 07: Train Loss = 0.4887, Val Loss = 0.4571
Epoch 08: Train Loss = 0.4123, Val Loss = 0.3905
Epoch 09: Train Loss = 0.3646, Val Loss = 0.3509
Epoch 10: Train Loss = 0.3365, Val Loss = 0.3063
Epoch 11: Train Loss = 0.2960, Val Loss = 0.2645
Epoch 12: Train Loss = 0.2438, Val Loss = 0.2363
Epoch 13: Train Loss = 0.2125, Val Loss = 0.2126
Epoch 14: Train Loss = 0.1887, Val Loss = 0.1869
Epoch 15: Train Loss = 0.1645, Val Loss = 0.1598
Epoch 16: Train Loss = 0.1385, Val Loss = 0.1355
Epoch 17: Train Loss = 0.1151, Val Loss = 0.1169
Epoch 18: Train Loss = 0.0970, Val Loss = 0.1045
Epoch 19: Train Loss = 0.0849, Val Loss = 0.0958
Epoch 20: Train Loss = 0.0754, Val Loss = 0.0865
Epoch 21: Train Loss

<All keys matched successfully>

In [29]:
#Caonfidence Prediction Analysis

# Reverse relation index
idx_to_relation = {idx: relation for relation, idx in relation_to_idx.items()}

# Get prediction confidences
model.eval()
with torch.no_grad():
    test_output = model(test_data)
    test_probs = torch.exp(test_output)
    test_predictions = torch.argmax(test_output, dim=1).cpu().numpy()
    test_confidence = torch.max(test_probs, dim=1)[0].cpu().numpy()

# Add to test_df
confidence_df = test_df.copy()
confidence_df['predicted_relation'] = [idx_to_relation[pred] for pred in test_predictions]
confidence_df['prediction_confidence'] = test_confidence
confidence_df['correct'] = confidence_df['predicted_relation'] == confidence_df['Relation']

# Plot confidence histogram
plt.figure(figsize=(10, 6))
sns.histplot(confidence_df['prediction_confidence'], bins=30, kde=True)
plt.title('Distribution of Prediction Confidence')
plt.xlabel('Confidence Score')
plt.ylabel('Frequency')
plt.axvline(x=confidence_df['prediction_confidence'].mean(), color='r', linestyle='--', 
            label=f'Mean: {confidence_df["prediction_confidence"].mean():.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('output/confidence_distribution.png', dpi=300)
plt.close()

# Plot confidence for correct vs. incorrect predictions
plt.figure(figsize=(10, 6))
sns.histplot(data=confidence_df, x='prediction_confidence', hue='correct', 
             bins=30, kde=True, common_norm=False, stat='density')
plt.title('Confidence Distribution: Correct vs. Incorrect Predictions')
plt.xlabel('Confidence Score')
plt.ylabel('Density')
plt.tight_layout()
plt.savefig('output/confidence_by_correctness.png', dpi=300)
plt.close()
print("Saved confidence by correctness plot to output/confidence_by_correctness.png")

In [30]:
#tSNE 

# Get embeddings
model.eval()
with torch.no_grad():
    device = next(model.parameters()).device
    entity_embeddings = model.entity_embedding(torch.arange(len(entity_to_idx), device=device)).cpu().numpy()

# Reverse entity idx
idx_to_entity = {idx: entity for entity, idx in entity_to_idx.items()}

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(entity_to_idx)-1))
embeddings_2d = tsne.fit_transform(entity_embeddings)

# Scatter plot
plt.figure(figsize=(12, 10))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.7, s=10)

# Annotate top 20 entities
edge_counts = {}
for entity_name in idx_to_entity.values():
    edge_counts[entity_name] = edge_counts.get(entity_name, 0) + 1

top_entities = sorted(edge_counts.items(), key=lambda x: x[1], reverse=True)[:20]
top_entity_indices = [entity_to_idx[entity] for entity, _ in top_entities if entity in entity_to_idx]

for idx in top_entity_indices:
    plt.annotate(
        idx_to_entity[idx],
        xy=(embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
        xytext=(5, 2),
        textcoords='offset points',
        fontsize=8
    )

plt.title('t-SNE Visualization of Entity Embeddings')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.tight_layout()
plt.savefig('output/entity_embeddings_tsne.png', dpi=300)
plt.close()


In [25]:
confidences = confidence_df['prediction_confidence'].values
correctness = confidence_df['correct'].values

n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_indices = np.digitize(confidences, bin_edges) - 1
bin_indices = np.clip(bin_indices, 0, n_bins - 1)

bin_accs = np.zeros(n_bins)
bin_confs = np.zeros(n_bins)
bin_counts = np.zeros(n_bins)

for i in range(len(confidences)):
    bin_idx = bin_indices[i]
    bin_accs[bin_idx] += correctness[i]
    bin_confs[bin_idx] += confidences[i]
    bin_counts[bin_idx] += 1

non_empty_bins = bin_counts > 0
bin_accs[non_empty_bins] /= bin_counts[non_empty_bins]
bin_confs[non_empty_bins] /= bin_counts[non_empty_bins]

plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
plt.plot(bin_confs[non_empty_bins], bin_accs[non_empty_bins], 'o-', label='Model')
plt.xlabel('Mean Predicted Confidence')
plt.ylabel('Accuracy')
plt.title('Calibration Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('output/calibration_curve.png', dpi=300)
plt.close()




In [33]:
# Create test set
pos_edges = test_data.edge_index.t().cpu()
pos_rel = test_data.edge_attr.cpu()
pos_labels = torch.ones(pos_edges.size(0))

neg_edges = []
neg_rel = []
existing_edges = set((src.item(), dst.item()) for src, dst in pos_edges)

num_neg = pos_edges.size(0)
while len(neg_edges) < num_neg:
    src = random.randint(0, test_data.num_nodes - 1)
    dst = random.randint(0, test_data.num_nodes - 1)
    if src != dst and (src, dst) not in existing_edges:
        neg_edges.append((src, dst))
        neg_rel.append(random.randint(0, len(relation_to_idx) - 1))
        existing_edges.add((src, dst))

neg_edges = torch.tensor(neg_edges, dtype=torch.long)
neg_rel = torch.tensor(neg_rel, dtype=torch.long)
neg_labels = torch.zeros(neg_edges.size(0))

test_edges = torch.cat([pos_edges, neg_edges], dim=0)
test_rel = torch.cat([pos_rel, neg_rel], dim=0)
test_labels = torch.cat([pos_labels, neg_labels], dim=0)

perm = torch.randperm(test_edges.size(0))
test_edges = test_edges[perm]
test_rel = test_rel[perm]
test_labels = test_labels[perm]

# Run prediction
model.eval()
predictions = []

device = next(model.parameters()).device
with torch.no_grad():
    for i in range(0, len(test_edges), 100):
        batch_edges = test_edges[i:i+100]
        batch_rel = test_rel[i:i+100]
        temp_data = Data(
            edge_index=batch_edges.t().to(device),
            edge_attr=batch_rel.to(device),
            num_nodes=test_data.num_nodes
        )
        output = model(temp_data)
        batch_preds = torch.exp(output).max(dim=1)[0].cpu().numpy()
        predictions.extend(batch_preds)

predictions = np.array(predictions)
labels = test_labels.numpy()

# Metrics
auroc = roc_auc_score(labels, predictions)
auprc = average_precision_score(labels, predictions)

# Plot ROC
fpr, tpr, _ = roc_curve(labels, predictions)

plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, label=f'ROC Curve (AUROC = {auroc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.savefig('output/link_prediction_roc.png', dpi=300)
plt.close()
